# ERA5 daily relative humidity: download and build GeoTIFFs (output: `F:\new_rh`)

Downloads ERA5 2 m temperature and 2 m dewpoint from the Copernicus Climate Data Store (CDS), computes **daily relative humidity** with the Magnus formula (same constants as `rh_from_td` in Notebook A), clips it to the county, and writes **one single-band GeoTIFF per day**:

`F:\new_rh\<County>\daily_tifs\<County>_RH_<YYYY-MM-DD>.tif`

The names and layout are the ones Notebook A already reads, so the files can be dropped into `...\DAILY_RH\RH_<COUNTY>\daily_tifs` later. Nothing outside `F:\new_rh` is ever written or changed.

**Two methods (set `RH_METHOD` in the configuration cell)**

| `RH_METHOD` | What is computed | CDS dataset |
|---|---|---|
| `"daily_mean"` (default) | RH from the **daily-mean** temperature and dewpoint | `derived-era5-single-levels-daily-statistics` |
| `"hourly_mean"` | RH from each of the 24 **hourly** values, then averaged over the day | `reanalysis-era5-single-levels` |

The two differ by a few % points (RH is non-linear in temperature; the hourly average is the higher one, most visibly with a large day/night temperature swing). **Use the same method for every county and every year.** If your Kakamega and Kilifi RH files were built from hourly values, use `"hourly_mean"` here.

**Before you run:** you need a CDS account. The notebook asks once for your API token and saves it, and tests the connection before downloading. The only manual step is accepting the dataset licence on the CDS website (once; the test cell prints the link if it is missing). The download is cached: if it stops, run the download cell again and it continues.

## 0. Install (run once, then comment out; restart the kernel afterwards if pip installed anything)

In [ ]:
%pip install cdsapi xarray netCDF4 dask geopandas rioxarray rasterio matplotlib shapely tqdm

## 1. Imports

In [ ]:
import os, re, glob, time, calendar, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd, xarray as xr
import geopandas as gpd
import rioxarray  # noqa: F401  (registers the .rio accessor)
import rasterio
import cdsapi
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Configuration

In [ ]:
# --- counties (shapefile) ---
SHAPEFILE_PATH = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"
NAME_COLUMN    = "adm1_name"
COUNTIES       = ["Kakamega", "Kilifi", "Nyeri"]              # e.g. ["Kakamega", "Kilifi", "Nyeri"]; the download box covers all listed counties

# --- period (inclusive). 2026-09-30 = last complete month used by Notebook A ---
START_DATE = "2026-01-01"
END_DATE   = "2026-11-09"

# --- method: "daily_mean" or "hourly_mean" (see the table above) ---
RH_METHOD = "daily_mean"

MAX_PARALLEL_REQUESTS = 4               # CDS requests queued at once; lower to 2 if CDS rejects extra requests

# --- output ---
OUT_DIR     = r"F:\new_rh"
RAW_DIR     = os.path.join(OUT_DIR, "raw_monthly")      # one small .nc per month (and variable)
STATS_DIR   = os.path.join(OUT_DIR, "stats_daily")      # county-mean RH per day (csv)
SAVE_NETCDF = True                                      # also one NetCDF per county

# --- optional: existing RH files to compare the new 2026 values with (skipped if the folder is not there) ---
COMPARE_ROOT = r"F:\VI_prediction\HISTORICAL WEATHER\DAILY_RH"      # <root>\RH_<COUNTY>\daily_tifs\<County>_RH_<date>.tif

assert RH_METHOD in ("daily_mean", "hourly_mean"), "RH_METHOD must be 'daily_mean' or 'hourly_mean'"
for d in (OUT_DIR, RAW_DIR, STATS_DIR):
    os.makedirs(d, exist_ok=True)
print("method:", RH_METHOD, "| output:", OUT_DIR)

## 3. Counties and the download box
The request box is the bounding box of the chosen counties plus 0.5 degree, snapped to the 0.25 degree ERA5 grid.

In [ ]:
gdf = gpd.read_file(SHAPEFILE_PATH)
assert NAME_COLUMN in gdf.columns, f"column {NAME_COLUMN!r} not found; available: {list(gdf.columns)}"
gdf = gdf.set_crs("EPSG:4326") if gdf.crs is None else gdf.to_crs("EPSG:4326")
gdf["_n"] = gdf[NAME_COLUMN].astype(str).str.strip().str.title()
missing = [c for c in COUNTIES if c.title() not in set(gdf["_n"])]
assert not missing, f"{missing} not found in {NAME_COLUMN!r}; available: {sorted(gdf['_n'].unique())}"

COUNTY_GDFS = {c: gdf[gdf["_n"] == c.title()].drop(columns="_n").copy() for c in COUNTIES}
sel = gdf[gdf["_n"].isin([c.title() for c in COUNTIES])]
w, s, e, n = sel.total_bounds
PAD = 0.5
AREA = [float(np.ceil((n + PAD) * 4) / 4), float(np.floor((w - PAD) * 4) / 4),
        float(np.floor((s - PAD) * 4) / 4), float(np.ceil((e + PAD) * 4) / 4)]      # N, W, S, E
print("CDS request area [N, W, S, E]:", AREA)
sel.plot(figsize=(4, 4), edgecolor="k"); plt.title(", ".join(COUNTIES)); plt.show()

## 4. Months and days to fetch

In [ ]:
cutoff = pd.Timestamp(END_DATE)
PLAN = []        # (year, month, [day strings])
for p in pd.period_range(pd.Timestamp(START_DATE), cutoff, freq="M"):
    last = calendar.monthrange(p.year, p.month)[1]
    if (p.year, p.month) == (cutoff.year, cutoff.month):
        last = min(last, cutoff.day)
    PLAN.append((p.year, p.month, [f"{d:02d}" for d in range(1, last + 1)]))
N_DAYS = sum(len(d) for _, _, d in PLAN)
print(f"{len(PLAN)} months, {N_DAYS} days: {PLAN[0][0]}-{PLAN[0][1]:02d} .. {PLAN[-1][0]}-{PLAN[-1][1]:02d}")

## 4b. CDS key and connection test (automatic)
- If `~/.cdsapirc` (or the `CDSAPI_KEY` environment variable) does not exist, you are asked **once** for your CDS personal access token (copy it from your CDS profile page) and the file is written for you. After that nothing is asked again.
- A one-day test request is then sent for the chosen dataset. If it fails because the **licence has not been accepted**, the exact page is printed: open it, log in, tick the licence once, and run this cell again. (A licence can only be accepted on the website; code cannot do it for you.)

In [ ]:
import getpass
from pathlib import Path

DAILY_DATASET  = "derived-era5-single-levels-daily-statistics"
HOURLY_DATASET = "reanalysis-era5-single-levels"
VARIABLES = {"2m_temperature": "t2m", "2m_dewpoint_temperature": "d2m"}
CDS_URL = "https://cds.climate.copernicus.eu/api"

CDSRC = Path.home() / ".cdsapirc"
if not CDSRC.exists() and not os.environ.get("CDSAPI_KEY"):
    key = getpass.getpass("CDS personal access token (CDS website > your name > Profile > API token): ").strip()
    CDSRC.write_text(f"url: {CDS_URL}\nkey: {key}\n")
    print("credentials saved to", CDSRC)
else:
    print("CDS credentials found")

def cds_preflight():
    ds_name = DAILY_DATASET if RH_METHOD == "daily_mean" else HOURLY_DATASET
    small = [0.5, 36.5, 0.0, 37.0]
    if RH_METHOD == "daily_mean":
        req = {"product_type": "reanalysis", "variable": ["2m_temperature"], "year": "2020", "month": "01", "day": ["01"],
               "daily_statistic": "daily_mean", "time_zone": "utc+00:00", "frequency": "1_hourly", "area": small, "format": "netcdf"}
    else:
        req = {"product_type": "reanalysis", "variable": ["2m_temperature"], "year": "2020", "month": "01", "day": ["01"],
               "time": ["00:00"], "area": small, "data_format": "netcdf", "download_format": "unarchived"}
    tmp = os.path.join(RAW_DIR, "_preflight.nc")
    try:
        cdsapi.Client(quiet=True).retrieve(ds_name, req, tmp)
        print(f"OK: CDS connection and licence work for {ds_name}")
    except Exception as ex:
        msg = str(ex)
        if "licen" in msg.lower():
            print(f"LICENCE NOT ACCEPTED for {ds_name}. Open this page, log in, scroll to 'Terms of use' and accept once:\n"
                  f"  https://cds.climate.copernicus.eu/datasets/{ds_name}?tab=download\nthen run this cell again.")
        elif any(k in msg.lower() for k in ("401", "403", "unauthor", "authenticat", "invalid")):
            print("The key was rejected. Delete", CDSRC, "and run this cell again to enter a new token.")
        else:
            print("Test request failed:", msg[:300])
        raise
    finally:
        if os.path.exists(tmp): os.remove(tmp)

cds_preflight()

## 5. Download (parallel, cached)
- `daily_mean`: two requests per month (temperature, dewpoint), 162 files for 2020-01 to 2026-09.
- `hourly_mean`: one request per month with both variables and all 24 hours, 81 files.

If CDS reports that the latest days are not available yet, the last 3 days are dropped and the request is repeated (this only matters when `END_DATE` is very recent). A failed request is retried up to 5 times; re-run the cell to retry any that still fail.

In [ ]:
def raw_path(tag, y, m, ndays):
    return os.path.join(RAW_DIR, f"{tag}_{y}_{m:02d}_{ndays}d.nc")

def best_file(tag, y, m):
    # the file of that month with the most days (a month that has since grown replaces the shorter one)
    fs = []
    for f in glob.glob(os.path.join(RAW_DIR, f"{tag}_{y}_{m:02d}_*d.nc")):
        k = re.search(r"_(\d+)d\.nc$", f)
        if k: fs.append((int(k.group(1)), f))
    return max(fs)[1] if fs else None

def _unavailable(msg):
    msg = msg.lower()
    return ("available yet" in msg) or ("latest date available" in msg) or ("not available" in msg)

def retrieve(dataset, request_fn, tag, y, m, days):
    days = list(days)
    for attempt in range(1, 6):
        dest = raw_path(tag, y, m, len(days))
        if os.path.exists(dest):
            return dest
        try:
            cdsapi.Client(quiet=True).retrieve(dataset, request_fn(days), dest + ".part")
            os.replace(dest + ".part", dest)
            return dest
        except Exception as ex:
            if _unavailable(str(ex)) and len(days) > 3:
                days = days[:-3]; tqdm.write(f"{tag} {y}-{m:02d}: latest days not available yet -> {len(days)} days"); continue
            if attempt == 5: raise
            time.sleep(30 * attempt)

def daily_req(cds_var, y, m):
    return lambda days: {"product_type": "reanalysis", "variable": [cds_var], "year": str(y), "month": f"{m:02d}", "day": days,
                         "daily_statistic": "daily_mean", "time_zone": "utc+00:00", "frequency": "1_hourly",
                         "area": AREA, "format": "netcdf"}

def hourly_req(y, m):
    return lambda days: {"product_type": "reanalysis", "variable": list(VARIABLES), "year": str(y), "month": f"{m:02d}", "day": days,
                         "time": [f"{h:02d}:00" for h in range(24)], "area": AREA,
                         "data_format": "netcdf", "download_format": "unarchived"}

jobs = []
for y, m, days in PLAN:
    if RH_METHOD == "daily_mean":
        for var, short in VARIABLES.items():
            if not os.path.exists(raw_path(f"{short}_daily_mean", y, m, len(days))):
                jobs.append((DAILY_DATASET, daily_req(var, y, m), f"{short}_daily_mean", y, m, days))
    else:
        if not os.path.exists(raw_path("hourly", y, m, len(days))):
            jobs.append((HOURLY_DATASET, hourly_req(y, m), "hourly", y, m, days))
print(f"{len(jobs)} request(s) still to download")

failed = []
with ThreadPoolExecutor(max_workers=MAX_PARALLEL_REQUESTS) as pool:
    futs = {pool.submit(retrieve, *j): j for j in jobs}
    for f in tqdm(as_completed(futs), total=len(futs), desc="CDS", unit="file"):
        try: f.result()
        except Exception as ex:
            failed.append((futs[f][2:5], str(ex)[:200])); tqdm.write(f"FAILED {futs[f][2:5]}: {str(ex)[:200]}")
print("finished.", f"{len(failed)} failed: run this cell again to retry." if failed else "All files present.")

## 6. Daily relative humidity
`RH = 100 * es(dewpoint) / es(temperature)`, with the Magnus saturation pressure (17.625, 243.04), clipped to 0-100. With `hourly_mean` the RH is computed for each hour and then averaged per UTC day.

In [ ]:
def es(tc):                      # saturation vapour pressure (hPa), Magnus; same constants as rh_from_td in Notebook A
    return 6.1094 * np.exp(17.625 * tc / (tc + 243.04))

def open_files(tag):
    files = [best_file(tag, y, m) for y, m, _ in PLAN]
    missing = [f"{y}-{m:02d}" for (y, m, _), f in zip(PLAN, files) if f is None]
    assert not missing, f"{tag}: months not downloaded: {missing[:6]}{' ...' if len(missing) > 6 else ''}: run the download cell again"
    ds = xr.open_mfdataset(files, combine="by_coords")
    ds = ds.drop_vars([v for v in ("number", "expver") if v in ds.coords and v not in ds.dims])
    if "expver" in ds.dims:                         # recent ERA5T months can carry two experiment versions
        ds = ds.mean("expver", skipna=True)
    tdim = "valid_time" if "valid_time" in ds.dims else "time"
    return ds.sortby(tdim), tdim

if RH_METHOD == "daily_mean":
    t_ds, tdim = open_files("t2m_daily_mean"); d_ds, _ = open_files("d2m_daily_mean")
    t = t_ds[[v for v in t_ds.data_vars][0]] - 273.15
    d = d_ds[[v for v in d_ds.data_vars][0]] - 273.15
    rh = (100 * es(d) / es(t)).clip(0, 100)
else:
    ds, tdim = open_files("hourly")
    t, d = ds["t2m"] - 273.15, ds["d2m"] - 273.15
    rh_h = (100 * es(d) / es(t)).clip(0, 100)
    rh = rh_h.resample({tdim: "1D"}).mean()

rh = rh.rename({tdim: "time"}) if tdim != "time" else rh
rh["time"] = pd.DatetimeIndex(rh["time"].values).normalize()
rh.name = "relative_humidity"; rh.attrs.update(units="%", long_name=f"2 m relative humidity, daily ({RH_METHOD})")
lon = "longitude" if "longitude" in rh.dims else "lon"; lat = "latitude" if "latitude" in rh.dims else "lat"
if float(rh[lon].max()) > 180:
    rh = rh.assign_coords({lon: ((rh[lon] + 180) % 360) - 180})
rh = rh.sortby(lon).sortby(lat, ascending=False)           # north-up, so the GeoTIFF is not flipped
rh_ds = rh.to_dataset().rio.set_spatial_dims(x_dim=lon, y_dim=lat).rio.write_crs("EPSG:4326")
n_t = rh.sizes["time"]
print(f"{n_t} days, {str(rh['time'].values[0])[:10]} .. {str(rh['time'].values[-1])[:10]} (expected {N_DAYS})")
assert n_t == N_DAYS, "the number of days differs from the plan: a month is missing or has fewer days than requested"

## 7. Clip to each county and write one GeoTIFF per day
Files that already exist are skipped, so the cell can be re-run. To refresh the newest days (ERA5 revises recent data), delete those files first.

In [ ]:
SERIES = {}
for county in COUNTIES:
    geom = COUNTY_GDFS[county].geometry
    clipped = rh_ds.rio.clip(geom.values, geom.crs, drop=True, all_touched=True)
    if SAVE_NETCDF:
        clipped.to_netcdf(os.path.join(OUT_DIR, f"{county}_relative_humidity_daily_{START_DATE[:4]}_{END_DATE[:4]}.nc"))
    v = clipped["relative_humidity"]
    y_dim, x_dim = v.rio.y_dim, v.rio.x_dim
    v = v.transpose("time", y_dim, x_dim).load()
    tif_dir = os.path.join(OUT_DIR, county, "daily_tifs"); os.makedirs(tif_dir, exist_ok=True)
    transform, crs, (h, wd) = v.rio.transform(), v.rio.crs, (v.sizes[y_dim], v.sizes[x_dim])
    written = skipped = 0
    for i in tqdm(range(v.sizes["time"]), desc=f"{county} GeoTIFFs", unit="day"):
        day = str(np.datetime_as_string(v["time"].values[i], unit="D"))
        p = os.path.join(tif_dir, f"{county}_RH_{day}.tif")
        if os.path.exists(p): skipped += 1; continue
        with rasterio.open(p, "w", driver="GTiff", height=h, width=wd, count=1, dtype="float32", crs=crs, transform=transform,
                           nodata=np.nan, compress="deflate") as dst:
            dst.write(v.isel(time=i).values.astype("float32"), 1)
            dst.set_band_description(1, day)
            dst.update_tags(DATE=day, VARIABLE=f"2m relative humidity (daily, %), method={RH_METHOD}")
        written += 1
    ser = v.mean(dim=[y_dim, x_dim]).to_series(); ser.index = pd.DatetimeIndex(ser.index).normalize()
    SERIES[county] = ser
    ser.rename("rh_pct").rename_axis("date").to_csv(os.path.join(STATS_DIR, f"{county}_RH_daily.csv"))
    print(f"{county}: {written} written, {skipped} already existed | grid {h} x {wd} | {tif_dir}")

## 8. Checks
**8a.** The county-mean RH per month and year, and the spread between years. Real RH differs between years; a spread below 0.5 % points in any month means something is wrong (a repeated year).

In [ ]:
for county, ser in SERIES.items():
    t = ser.groupby([ser.index.month, ser.index.year]).mean().unstack(); t.index.name = "month"
    full = [c for c in t.columns if c < ser.index.year.max()]
    spread = t[full].std(axis=1)
    print(f"\n=== {county}: monthly mean RH (%) by year ===")
    print(t.round(1).to_string())
    print("\nspread between years (sd, % points):", spread.round(1).to_dict())
    print("!! some months have almost no year-to-year spread: do not use these files" if (spread < 0.5).any() else "OK: every month varies between years")
    fig, ax = plt.subplots(figsize=(11, 3)); ax.plot(ser.index, ser.values, lw=0.7)
    ax.set_title(f"{county}: daily mean RH (%), {RH_METHOD}"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

**8b.** The same scan as before, on the files just written: no file may be identical to the same day of the first year.

In [ ]:
def rd(p):
    with rasterio.open(p) as s: return s.read(1, masked=True).astype("float32").filled(np.nan)
for county in COUNTIES:
    d_ = os.path.join(OUT_DIR, county, "daily_tifs"); rows = []
    for m in range(1, 13):
        base = None
        for y in range(pd.Timestamp(START_DATE).year, pd.Timestamp(END_DATE).year + 1):
            p = os.path.join(d_, f"{county}_RH_{y}-{m:02d}-15.tif")
            if not os.path.exists(p): continue
            a = rd(p)
            if base is None: base = a; continue
            rows.append(float(np.nanmax(np.abs(a - base))))
    same = sum(r < 1e-6 for r in rows)
    print(f"{county}: {len(rows)} comparisons, {same} identical to the first year's file" + ("  <-- PROBLEM" if same else "  (OK)"))
    n_files = len(glob.glob(os.path.join(d_, f"{county}_RH_*.tif")))
    print(f"{county}: {n_files} daily files (expected {N_DAYS})")

**8c (optional).** Compare the new 2026 values with existing RH files, if the folder is there. A mean difference near 0 (within about 1 % point) means the existing files used the same method. A clearly different mean means they used the other one: re-run with the other `RH_METHOD`, or keep the new files for the whole series so all years use one method.

In [ ]:
for county, ser in SERIES.items():
    old_dir = os.path.join(COMPARE_ROOT, f"RH_{county.upper()}", "daily_tifs")
    if not os.path.isdir(old_dir):
        print(f"{county}: no existing folder at {old_dir}: comparison skipped"); continue
    rows = []
    for day, new in ser[ser.index.year == ser.index.year.max()].items():
        p = os.path.join(old_dir, f"{county}_RH_{day:%Y-%m-%d}.tif")
        if not os.path.exists(p): continue
        a = rd(p)
        if np.nanmax(a) <= 1.5: a = a * 100
        rows.append((float(np.nanmean(a)), float(new)))
    if not rows: print(f"{county}: no existing files for {ser.index.year.max()}"); continue
    old, new = np.array(rows).T; dlt = new - old
    print(f"{county}: {len(rows)} days | mean(new - existing) = {dlt.mean():+.2f} % | mean |diff| = {np.abs(dlt).mean():.2f} % | max |diff| = {np.abs(dlt).max():.2f} %")

In [ ]:
import re, os
_pat = re.compile(r"^(.+_RH_)(\d{4})-(\d{2})-(\d{2})(\.\w+)?$")
for county in COUNTIES:
    d_ = os.path.join(OUT_DIR, county, "daily_tifs")
    if not os.path.isdir(d_): print(f"{county}: no folder {d_}"); continue
    renamed = clash = 0
    for f in os.listdir(d_):
        m = _pat.match(f)
        if not m: continue
        new = f"{m.group(1)}{m.group(2)}_{m.group(3)}_{m.group(4)}{m.group(5) or ''}"
        if os.path.exists(os.path.join(d_, new)): clash += 1; continue
        os.rename(os.path.join(d_, f), os.path.join(d_, new)); renamed += 1
    left = len([f for f in os.listdir(d_) if _pat.match(f)])
    print(f"{county}: {renamed} renamed, {clash} skipped (target exists), {left} still hyphenated | {len(os.listdir(d_))} files in {d_}")

## 9. After this notebook
The new files are in `F:\new_rh\<County>\daily_tifs`. Nothing else has been touched. To use them:

1. **Back up** the existing folder `...\DAILY_RH\RH_<COUNTY>\daily_tifs` (rename it), and copy the new `daily_tifs` folder in its place.
2. **SEAS5 bias correction:** `REUSE_OLD_RECORDS = False`; delete `FORECASTED_WEATHER_BC_V2\_bias_cache`; delete the existing `humidity\rh` outputs of that county in `FORECASTED_WEATHER_BC_V2` (`SKIP_EXISTING` would keep them); re-run.
3. **DEM correction:** delete the county's `humidity\rh` outputs in `FORECASTED_WEATHER_30M_BC_V2`; re-run.
4. **Notebook A:** `WX_VERSION = "v3"`; re-run sections 2, 5 and 6, then the clipping cell 7b.
5. **Notebook A2:** `REBUILD_COARSE = True` (otherwise the cached coarse history keeps the old RH); re-run sections 4 to 7.
6. **Notebook B, then C.** B's data guard should now print "RH year-to-year spread OK" for the county.